### Implementing the vector search process using persistent index


In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer('all-MiniLM-L6-v2')

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [7]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

Note that while we did not need to do the ingestion step, we still had to do the encoding (vector embedding).

In [8]:
results

[{'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```',
  'doc_id': '5ca6890c1a'},
 {'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BOOTSTRAP_SERVERS_CO

In [9]:
# filter by course

results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [10]:
results

[{'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Project: what does "reproducibility" mean — do reviewers need access to my API keys?',
  'answer': "Never share API keys or hosted-service credentials in your repo. Reproducibility means a peer reviewer can clone the repo and follow your README to recreate the system from scratch — using their own credentials.\n\nConcretely:\n\n- Provide a script (or notebook) that ingests the dataset and (re)builds the search index locally.\n- Ship a `.env.example` with the variable names but no values; have the reviewer create their own `.env` with their own keys. Keep `.env` in `.gitignore`.\n- Use a cheap model (`gpt-4o-mini`, Groq, etc.) so reviewers don't burn through credits when running your project.\n- Pin dependency versions (`requirements.txt` / `pyproject.toml` lock file) and document the Python version (and Docker version, if used).",
  'doc_id': 'e5d8a2c761'},
 {'course': 'llm-zoomcamp',
  'section': 'Module 1: Ag

### Using sqlitesearch vector search in RAG
Let's use our persistent vector index in RAG.

In [11]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [12]:
vector_assistant.rag("the program has already begun, can I still sign up?")

"Yes, you can still join. You don’t need to wait for a confirmation email—you're accepted, and you can start learning and submitting homework while the form is open."

In [13]:
vs_index.close()

### Comparing minsearch and sqlitesearch for vector search

Here is how the two approaches compare:

* minsearch ```VectorSearch```: in-memory (numpy), exact cosine similarity, must re-compute embeddings on startup, good for experiments and notebooks
* sqlitesearch ```VectorSearchIndex```: persistent (SQLite ```.db``` file), ANN (LSH/IVF/HNSW) with exact rerank, can open an existing index, good for projects and persistence

Regarding ```sqlitesearch```, this is not something you would use for commercial deployment, since it was created for educational purposes.  However, for coursework, it is useful:  SQLite is a lightweight database and many hosts have a free SQLite database option that may work well for our purposes.